# Proyecto_01

## Objetivos

### Generales
Predecir e interpretar el desempeño en el examen de estudiantes (`Exam_Score`) a partir de factores académicos, personales, familiares y escolares, mediante la comparación de tres especificaciones de regresión (lineal, polinomial y con interacciones) ajustadas en cuatro versiones (sin penalización, Ridge, Lasso y ElasticNet), para identificar qué especificación representa mejor las relaciones presentes en los datos, qué método de regularización generaliza mejor y qué factores presentan mayor asociación predictiva con el desempeño.

### Específicos
1. **Analizar el conjunto de datos** `StudentPerformanceFactors.csv` en cuanto a su origen, contenido, calidad y estructura, identificando valores faltantes, valores fuera de rango, tipos de variables y posibles transformaciones necesarias para su uso en modelos de regresión.

2. **Construir un pipeline de preprocesamiento** que incluya limpieza de datos, imputación de valores faltantes, codificación de variables categóricas (ordinales y binarias), escalamiento de variables numéricas y partición de entrenamiento y prueba, garantizando que todas las transformaciones se ajusten únicamente con los datos de entrenamiento.

3. **Definir tres especificaciones de regresión**: una lineal múltiple sin interacciones, una polinomial con términos seleccionados y una con interacciones justificadas entre factores, cada una en cuatro versiones: sin penalización (OLS), Ridge, Lasso y ElasticNet.

4. **Seleccionar los hiperparámetros de penalización** de las versiones Ridge, Lasso y ElasticNet mediante validación cruzada de 5 folds con `GridSearchCV`, usando R² como métrica de optimización.

5. **Ajustar y comparar los doce modelos resultantes** mediante métricas de desempeño (R², RMSE) en entrenamiento y prueba, con visualizaciones que permitan identificar diferencias entre especificaciones y entre versiones de penalización.

6. **Realizar un análisis de residuos** sobre el modelo lineal sin penalización para evaluar los seis supuestos clásicos de la regresión: no linealidad, correlación en errores, varianza no constante, outliers, puntos palanca y colinealidad, y proponer las transformaciones pertinentes a partir de los resultados.

7. **Analizar la significancia de los factores** en el modelo lineal sin penalización, mediante pruebas t individuales y la prueba F global, identificando qué factores se asocian significativamente con `Exam_Score`, y reportando el signo, magnitud e intervalos de confianza de los coeficientes estimados.

8. **Comparar los resultados entre las tres especificaciones y las cuatro versiones** para determinar qué combinación logra el mejor equilibrio entre capacidad predictiva, interpretabilidad y estabilidad, y **concluir respecto al objetivo general** sobre qué factores y qué tipo de modelo describen mejor el desempeño en el examen.

## Marco Teórico

### Regresión lineal

La regresión lineal es un método de aprendizaje supervisado utilizado para predecir una variable cuantitativa a partir de una o más variables predictoras. En este proyecto, la variable respuesta será la calificación obtenida en el examen, mientras que las variables predictoras serán los factores disponibles en el conjunto de datos *Student Performance Factor*.

En su forma múltiple, el modelo se expresa como:

$$
Y_i = \beta_0 + \beta_1X_{i1} + \beta_2X_{i2} + \cdots + \beta_pX_{ip} + \varepsilon_i
$$

donde:

- $Y_i$ es la calificación observada del estudiante $i$.
- $\beta_0$ es el intercepto.
- $\beta_j$ representa el cambio promedio esperado en $Y$ asociado con un incremento de una unidad en $X_j$, manteniendo constantes los demás factores.
- $\varepsilon_i$ representa el error aleatorio no explicado por el modelo.

Los coeficientes se estiman habitualmente mediante mínimos cuadrados ordinarios (OLS), que minimizan la suma de cuadrados de los residuos:

$$
RSS = \sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

El residuo $e_i=y_i-\hat{y}_i$ es la diferencia entre la calificación observada y la predicha. Un modelo con residuos pequeños tiende a ajustarse mejor a los datos de entrenamiento, aunque el desempeño debe verificarse también con datos no utilizados durante el ajuste.

Para evaluar el ajuste pueden emplearse métricas como el error cuadrático medio (MSE), la raíz del error cuadrático medio (RMSE), el error absoluto medio (MAE) y el coeficiente de determinación $R^2$. El valor de $R^2$ representa la proporción de variabilidad de la respuesta explicada por el modelo; sin embargo, debe complementarse con métricas de prueba o validación cruzada para evitar conclusiones basadas únicamente en el entrenamiento.

### Regresión polinomial

La regresión polinomial permite modelar relaciones no lineales entre una variable predictora y la respuesta, manteniendo un modelo lineal respecto a sus coeficientes. Por ejemplo, un modelo cuadrático para una variable $X$ se expresa como:

$$
Y_i = \beta_0 + \beta_1X_i + \beta_2X_i^2 + \varepsilon_i
$$

De forma general, un polinomio de grado $d$ es:

$$
Y_i = \beta_0 + \beta_1X_i + \beta_2X_i^2 + \cdots + \beta_dX_i^d + \varepsilon_i
$$

Aunque la relación entre $X$ y $Y$ puede ser curva, el modelo sigue siendo lineal porque los parámetros $\beta_j$ se estiman linealmente. Este enfoque es útil si los gráficos de dispersión o de residuos muestran curvatura que una regresión lineal simple no puede representar.

Un grado polinomial alto aumenta la flexibilidad del modelo, pero también el riesgo de sobreajuste: el modelo puede aprender ruido específico de los datos de entrenamiento y generalizar mal. Por ello, el grado debe elegirse mediante validación cruzada y comparando el error de prueba.

### Interacción de factores

Existe una interacción cuando el efecto de una variable predictora sobre la calificación depende del nivel de otra variable. Por ejemplo, el efecto de las horas de estudio podría ser distinto según las sesiones de tutoria del estudiante o el acceso a recursos educativos.

Para dos predictores $X_1$ y $X_2$, un modelo con interacción es:

$$
Y_i = \beta_0 + \beta_1X_{i1} + \beta_2X_{i2} + \beta_3(X_{i1}X_{i2}) + \varepsilon_i
$$

El término $X_1X_2$ representa la interacción. Al reorganizar el modelo se obtiene:

$$
Y_i = \beta_0 + (\beta_1+\beta_3X_{i2})X_{i1} + \beta_2X_{i2} + \varepsilon_i
$$

Por tanto, el efecto de $X_1$ deja de ser constante: depende de $X_2$. Si el coeficiente $\beta_3$ es relevante, incluir solamente los efectos principales puede producir una representación incompleta de la relación entre los factores y la calificación.

Debe respetarse el principio jerárquico: si se incorpora una interacción $X_1X_2$, también deben conservarse $X_1$ y $X_2$ en el modelo, aun cuando alguno de sus coeficientes individuales no resulte significativo.

### Significancia de factores

La significancia estadística permite evaluar si existe evidencia de que un predictor tiene asociación con la respuesta, después de controlar por los demás predictores incluidos en el modelo.

Para cada factor se plantea, usualmente, la prueba de hipótesis:

$$
H_0:\beta_j=0
$$

$$
H_a:\beta_j\neq0
$$

El estadístico de prueba es:

$$
t = \frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}
$$

donde $SE(\hat{\beta}_j)$ es el error estándar del coeficiente estimado. A partir de este estadístico se obtiene el *p-value*. Un *p-value* pequeño, comúnmente menor que $\alpha=0.05$, proporciona evidencia para rechazar $H_0$ y concluir que el factor está asociado con la calificación dentro del modelo planteado.

La significancia no equivale a causalidad ni garantiza una mejora importante en la predicción. Un factor puede tener un *p-value* pequeño y un efecto práctico reducido; también puede parecer no significativo cuando existe colinealidad con otros predictores. Por ello, la interpretación debe integrar el tamaño y signo del coeficiente, los intervalos de confianza, el conocimiento del problema y el desempeño predictivo.

La significancia global del modelo puede evaluarse mediante una prueba $F$, cuya hipótesis nula establece que todos los coeficientes de pendiente son iguales a cero:

$$
H_0:\beta_1=\beta_2=\cdots=\beta_p=0
$$

### Regularización

La regularización modifica la función de costo de mínimos cuadrados al agregar una penalización sobre el tamaño de los coeficientes. Su objetivo es disminuir la varianza del modelo, reducir el sobreajuste y tratar de forma más estable conjuntos con variables correlacionadas o con muchas características.

Antes de ajustar modelos penalizados, las variables numéricas deben estandarizarse, ya que las penalizaciones dependen de la escala de los coeficientes. La selección del hiperparámetro de penalización debe realizarse con validación cruzada sobre el conjunto de entrenamiento.

#### Ridge

La regresión Ridge utiliza una penalización $L2$:

$$
\min_{\beta}\left\{
RSS+\lambda\sum_{j=1}^{p}\beta_j^2
\right\}
$$

El parámetro $\lambda\geq0$ controla la intensidad de la penalización. Si $\lambda=0$, el modelo equivale a mínimos cuadrados ordinarios. Conforme $\lambda$ aumenta, los coeficientes se reducen hacia cero, pero normalmente no se vuelven exactamente cero.

Ridge es especialmente útil ante multicolinealidad, ya que estabiliza los coeficientes de predictores correlacionados. Conserva todas las variables, por lo que privilegia la predicción y la estabilidad sobre la selección explícita de factores.

#### Lasso

La regresión Lasso utiliza una penalización $L1$:

$$
\min_{\beta}\left\{
RSS+\lambda\sum_{j=1}^{p}|\beta_j|
\right\}
$$

A diferencia de Ridge, Lasso puede llevar algunos coeficientes exactamente a cero. Por ello realiza selección automática de variables y genera modelos más escasos e interpretables.

Lasso puede ser conveniente si se espera que solo una parte de los factores disponibles influya de forma relevante en el resultado del examen. No obstante, cuando varios predictores están altamente correlacionados, puede conservar uno y excluir otros de manera inestable.

#### ElasticNet

ElasticNet combina las penalizaciones Ridge y Lasso:

$$
\min_{\beta}\left\{
RSS+\lambda\left[
(1-\alpha)\frac{1}{2}\sum_{j=1}^{p}\beta_j^2+
\alpha\sum_{j=1}^{p}|\beta_j|
\right]
\right\}
$$

donde $\lambda$ controla la intensidad total de la penalización y $\alpha$ define la mezcla:

- $\alpha=0$: comportamiento equivalente a Ridge.
- $\alpha=1$: comportamiento equivalente a Lasso.
- $0<\alpha<1$: combinación de ambas penalizaciones.

ElasticNet es adecuado cuando existen grupos de variables correlacionadas, situación posible en factores académicos y socioeconómicos. Puede seleccionar variables, como Lasso, y al mismo tiempo distribuir de manera más estable el efecto entre predictores relacionados, como Ridge.

### Seis errores comunes en una regresión

1. **No linealidad entre predictores y respuesta.**  
   Ocurre cuando la relación real presenta curvatura y se ajusta únicamente una recta. Puede detectarse mediante gráficos de residuos; un patrón sistemático sugiere que el modelo lineal es insuficiente. Como respuesta, pueden evaluarse transformaciones, términos polinomiales o interacciones justificadas.

2. **Correlación entre términos de error.**  
   Los errores deberían ser independientes. Si las observaciones están ordenadas en el tiempo, pertenecen al mismo grupo o contienen registros relacionados, los errores pueden correlacionarse. Esto afecta los errores estándar, intervalos de confianza y pruebas de hipótesis.

3. **Varianza no constante de los errores.**  
   La heterocedasticidad ocurre cuando la dispersión de los residuos cambia con el nivel de las predicciones. Puede observarse como una forma de embudo en el gráfico de residuos. Es posible considerar transformaciones de la respuesta, métodos ponderados o estimadores robustos.

4. **Valores atípicos (*outliers*).**  
   Son observaciones con una respuesta muy alejada de lo predicho por el modelo. Pueden deberse a errores de captura, casos excepcionales o variables relevantes ausentes. Deben revisarse con residuos estudentizados, pero no eliminarse automáticamente: primero debe verificarse su origen.

5. **Puntos de alta palanca (*high leverage*).**  
   Son observaciones con valores inusuales en los predictores y que pueden modificar de manera importante la recta o el hiperplano de regresión. Se analizan mediante estadísticas de palanca y medidas de influencia, como la distancia de Cook.

6. **Colinealidad o multicolinealidad.**  
   Se presenta cuando dos o más predictores contienen información muy semejante. Dificulta separar su efecto individual, incrementa los errores estándar y vuelve inestables los coeficientes. Puede detectarse mediante correlaciones y el factor de inflación de varianza (VIF). Ridge y ElasticNet son alternativas útiles para mitigar sus efectos.

### Tema elegido para el proyecto

El tema del proyecto se define como la **predicción e interpretación del desempeño en examen de estudiantes mediante modelos de regresión regularizados**. El objetivo, como ya se menciono, es estimar la calificación de examen a partir de los 29 factores restantes del conjunto de datos *Student Performance Factor*, desarrollando un pipeline completo de preparación, modelado, evaluación e interpretación.

Se propone comparar tres especificaciones de regresión:

1. **Modelo lineal múltiple:** incorpora los factores originales disponibles.
2. **Modelo polinomial:** incorpora términos polinomiales seleccionados para representar relaciones no lineales plausibles.
3. **Modelo con interacciones:** incorpora interacciones justificadas entre factores académicos, de hábitos y de contexto del estudiante.

Cada especificación se ajustará en cuatro versiones:

| Versión | Método |
|---|---|
| Base | Regresión lineal sin penalización |
| Ridge | Penalización $L2$ |
| Lasso | Penalización $L1$ |
| ElasticNet | Penalización combinada $L1$ y $L2$ |

En total, se compararán doce modelos. El pipeline deberá incluir limpieza de datos, tratamiento de valores faltantes y variables categóricas, partición de entrenamiento y prueba, estandarización dentro del pipeline, ajuste de hiperparámetros mediante validación cruzada y evaluación final con métricas como RMSE, MAE y $R^2$.

La comparación permitirá responder tres preguntas centrales: qué especificación representa mejor las relaciones presentes en los datos, qué método de regularización logra la mejor capacidad de generalización y cuáles factores presentan mayor asociación predictiva con el desempeño en el examen.

## Análisis del Dataset

### Análisis
El conjunto de datos **StudentPerformanceFactors.csv** proviene de la plataforma [Kaggle](https://www.kaggle.com/datasets/fares279/student-performance-factors), donde se publica para analizar los factores académicos, personales, familiares y escolares asociados con el desempeño estudiantil. El archivo contiene 6,607 muestras y 20 variables: 19 predictores y la variable objetivo **Exam_Score**. Cada muestra representa a un estudiante y reúne información sobre sus hábitos de estudio, asistencia, resultados previos, descanso, tutorías, actividad física, motivación, acceso a internet y recursos, contexto familiar, características escolares y variables demográficas. De esta forma, las muestras permiten observar cómo la combinación de condiciones académicas, sociales y personales se relaciona con la calificación final del examen.

El análisis busca construir modelos de regresión capaces de predecir **Exam_Score** y comparar qué factores aportan mayor capacidad predictiva. Las variables numéricas son **Hours_Studied**, **Attendance**, **Sleep_Hours**, **Previous_Scores**, **Tutoring_Sessions** y **Physical_Activity**; estas se estandarizarán mediante **StandardScaler** de **sklearn** para que tengan una escala comparable. Este paso es especialmente importante en Ridge, Lasso y ElasticNet, pues sus penalizaciones dependen de la magnitud de los coeficientes y podrían penalizar de manera desigual a variables medidas en escalas distintas.

Las variables binarias **Extracurricular_Activities**, **Internet_Access**, **School_Type**, **Learning_Disabilities**, **Gender** y **Peer_Influence** se transformarán mediante *one-hot encoding*. Las variables ordinales **Parental_Involvement**, **Access_to_Resources**, **Motivation_Level**, **Family_Income**, **Teacher_Quality**, **Parental_Education_Level** y **Distance_from_Home** se codificarán mediante *ordinal encoding*, respetando un orden definido. Por ejemplo, `Low < Medium < High`, `High School < College < Postgraduate` y `Near < Moderate < Far`.

Al realizar la regresión, se espera que factores como **Previous_Scores**, **Attendance**, **Hours_Studied**, **Tutoring_Sessions**, **Access_to_Resources** y **Motivation_Level** se asocien positivamente con la calificación de examen, mientras que otros como **Learning_Disabilities** o **Extracurricular_Activities** tengan el efecto contrario. Sin embargo, el modelo permitirá cuantificar estas relaciones, identificar variables redundantes y comparar si la regularización mejora la generalización.

### Variables

| Variable | Tipo de variable | Definición | Valores esperados / categorías |
|---|---|---|---|
| `Hours_Studied` | Numérica | Horas de estudio por semana | 0 a 44 |
| `Attendance` | Numérica | Porcentaje de asistencia | 0 a 100 |
| `Parental_Involvement` | Categórica ordinal | Participación de los padres | `Low`, `Medium`, `High` |
| `Access_to_Resources` | Categórica ordinal | Acceso a recursos educativos | `Low`, `Medium`, `High` |
| `Extracurricular_Activities` | Categórica binaria | Participa en actividades extracurriculares | `No`, `Yes` |
| `Sleep_Hours` | Numérica | Horas promedio de sueño por noche | 4 a 10 |
| `Previous_Scores` | Numérica | Calificaciones previas | 50 a 100 |
| `Motivation_Level` | Categórica ordinal | Nivel de motivación | `Low`, `Medium`, `High` |
| `Internet_Access` | Categórica binaria | Disponibilidad de internet | `No`, `Yes` |
| `Tutoring_Sessions` | Numérica | Sesiones de tutoría por mes | 0 a 8 |
| `Family_Income` | Categórica ordinal | Nivel de ingreso familiar | `Low`, `Medium`, `High` |
| `Teacher_Quality` | Categórica ordinal | Calidad percibida del profesorado | `Low`, `Medium`, `High` |
| `School_Type` | Categórica binaria | Tipo de escuela | `Public`, `Private` |
| `Peer_Influence` | Categórica ordinal | Influencia de los pares | `Negative`, `Neutral`, `Positive` |
| `Physical_Activity` | Numérica | Horas de actividad física por semana | 0 a 6 |
| `Learning_Disabilities` | Categórica binaria | Presencia de discapacidad de aprendizaje | `No`, `Yes` |
| `Parental_Education_Level` | Categórica ordinal | Máximo nivel educativo de los padres | `High School`, `College`, `Postgraduate` |
| `Distance_from_Home` | Categórica ordinal | Distancia entre hogar y escuela | `Near`, `Moderate`, `Far` |
| `Gender` | Categórica binaria | Género del estudiante | `Female`, `Male` |
| `Exam_Score` | Numérica | Calificación final del examen; variable objetivo | 0 a 100 |

## Pipeline de Trabajo y Modelado

### Librerias

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import GridSearchCV

### Carga de Datos

Se cargan los datos directamente del repostirio de Github del proyecto usando el 'archivo crudo' del dataset.

In [2]:
#url = 'https://raw.githubusercontent.com/juanmoranerre/student-performance-regression/refs/heads/main/StudentPerformanceFactors.csv'
#df = pd.read_csv(url)
df = pd.read_csv('StudentPerformanceFactors.csv')
df.head(5)

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23.0,84.0,Low,High,No,7,73,Low,Yes,0.0,Low,Medium,Public,Positive,3.0,No,High School,Near,Male,67.0
1,19.0,64.0,Low,Medium,No,8,59,Low,Yes,NaN,Medium,Medium,Public,Negative,4.0,No,College,Moderate,Female,61.0
2,24.0,98.0,Medium,Medium,Yes,7,91,Medium,Yes,2.0,Medium,Medium,Public,Neutral,4.0,No,Postgraduate,Near,Male,74.0
3,29.0,89.0,Low,Medium,Yes,8,98,Medium,Yes,1.0,Medium,Medium,Public,Negative,4.0,No,High School,Moderate,Male,71.0
4,19.0,92.0,Medium,Medium,Yes,6,65,Medium,Yes,3.0,Medium,High,Public,Neutral,4.0,No,College,Near,Female,70.0


### Análisis Exploratorio de Datos

Previamente se realizo un análisis exploratorio para la realización del análisis del dataset. Sin embargo, a continuación se muestra un breve resumen del datset.

In [3]:
# Numero de filas y columnas
nrows, ncols = df.shape
print(f'El DataFrame tiene {nrows} filas y {ncols} columnas.')

# Tipos de datos y columnas
print('\nTipos de datos:')
print(df.dtypes)

El DataFrame tiene 6607 filas y 20 columnas.

Tipos de datos:
Hours_Studied                 float64
Attendance                    float64
Parental_Involvement           object
Access_to_Resources            object
Extracurricular_Activities     object
Sleep_Hours                     int64
Previous_Scores                 int64
Motivation_Level               object
Internet_Access                object
Tutoring_Sessions             float64
Family_Income                  object
Teacher_Quality                object
School_Type                    object
Peer_Influence                 object
Physical_Activity             float64
Learning_Disabilities          object
Parental_Education_Level       object
Distance_from_Home             object
Gender                         object
Exam_Score                    float64
dtype: object


In [4]:
# Valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())


Valores nulos por columna:
Hours_Studied                  2
Attendance                     2
Parental_Involvement           1
Access_to_Resources            2
Extracurricular_Activities     2
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               1
Internet_Access                1
Tutoring_Sessions              1
Family_Income                  1
Teacher_Quality               79
School_Type                    2
Peer_Influence                 1
Physical_Activity              3
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            68
Gender                         1
Exam_Score                     2
dtype: int64


In [5]:
# Valores Duplicados
print('Filas duplicadas:')
print(df.duplicated().sum())

Filas duplicadas:
0


In [6]:
# Descripción columnas numéricas
df.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6605.000000,6605.000000,6607.00000,6607.000000,6606.000000,6604.000000,6605.000000
mean,19.973808,79.948978,7.02906,75.070531,1.493642,2.967293,67.215291
std,5.995209,11.736506,1.46812,14.399784,1.230648,1.031015,4.217186
min,0.000000,-91.000000,4.00000,50.000000,0.000000,0.000000,-65.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [7]:
for i in df.columns:
    if df[i].dtype == 'object':
        print(f'Columna: {i}')
        print(df[i].unique())

Columna: Parental_Involvement
['Low' 'Medium' 'High' nan]
Columna: Access_to_Resources
['High' 'Medium' 'Low' nan]
Columna: Extracurricular_Activities
['No' 'Yes' nan]
Columna: Motivation_Level
['Low' 'Medium' 'High' nan]
Columna: Internet_Access
['Yes' 'No' nan]
Columna: Family_Income
['Low' 'Medium' 'High' nan]
Columna: Teacher_Quality
['Medium' 'High' 'Low' nan]
Columna: School_Type
['Public' 'Private' nan]
Columna: Peer_Influence
['Positive' 'Negative' 'Neutral' nan]
Columna: Learning_Disabilities
['No' 'Yes']
Columna: Parental_Education_Level
['High School' 'College' 'Postgraduate' nan]
Columna: Distance_from_Home
['Near' 'Moderate' 'Far' nan]
Columna: Gender
['Male' 'Female' nan]


Las columnas con valores invalidos son `Attendance` y `Exam_Score` por lo tanto eliminaremos del dataset estas filas invalidas, asi mismo no hay inconsitestencias de registro en las variables categóricas.

### Limpieza de datos

Los valores nulos los imputaremos con la mediana y las variables categóricas con la moda, así mismo elminamos los valores negativos primero haciendolos NA para despues eliminarlos.

In [8]:
df.loc[~df['Attendance'].between(0, 100), 'Attendance'] = pd.NA
df.loc[~df['Exam_Score'].between(0, 100), 'Exam_Score'] = pd.NA
df = df.dropna(subset=['Exam_Score']).reset_index(drop=True)

num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(exclude='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print(df.isna().sum())
df

Hours_Studied                 0
Attendance                    0
Parental_Involvement          0
Access_to_Resources           0
Extracurricular_Activities    0
Sleep_Hours                   0
Previous_Scores               0
Motivation_Level              0
Internet_Access               0
Tutoring_Sessions             0
Family_Income                 0
Teacher_Quality               0
School_Type                   0
Peer_Influence                0
Physical_Activity             0
Learning_Disabilities         0
Parental_Education_Level      0
Distance_from_Home            0
Gender                        0
Exam_Score                    0
dtype: int64


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23.0,84.0,Low,High,No,7,73,Low,Yes,0.0,Low,Medium,Public,Positive,3.0,No,High School,Near,Male,67.0
1,19.0,64.0,Low,Medium,No,8,59,Low,Yes,1.0,Medium,Medium,Public,Negative,4.0,No,College,Moderate,Female,61.0
2,24.0,98.0,Medium,Medium,Yes,7,91,Medium,Yes,2.0,Medium,Medium,Public,Neutral,4.0,No,Postgraduate,Near,Male,74.0
3,29.0,89.0,Low,Medium,Yes,8,98,Medium,Yes,1.0,Medium,Medium,Public,Negative,4.0,No,High School,Moderate,Male,71.0
4,19.0,92.0,Medium,Medium,Yes,6,65,Medium,Yes,3.0,Medium,High,Public,Neutral,4.0,No,College,Near,Female,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6598,25.0,69.0,High,Medium,No,7,76,Medium,Yes,1.0,High,Medium,Public,Positive,2.0,No,High School,Near,Female,68.0
6599,23.0,76.0,High,Medium,No,8,81,Medium,Yes,3.0,Low,High,Public,Positive,2.0,No,High School,Near,Female,69.0
6600,20.0,90.0,Medium,Low,Yes,6,65,Low,Yes,3.0,Low,Medium,Public,Negative,2.0,No,Postgraduate,Near,Female,68.0
6601,10.0,86.0,High,High,Yes,6,91,High,Yes,2.0,Low,Medium,Private,Positive,3.0,No,High School,Far,Female,68.0


Los puntos palanca se modificaran despues de el análisis de residuos.

### Ingeniería de Características

Se separan las columnas por su categoría para su transformación, esto se usa de forma generalizada para todos los modelos. Así mmismo se realiza una lista de listas de los valores posibles que puede tomar cada categoría para incluirlo en el preprocessor esto con el orden respectivo de la lista de categorías.

In [9]:
target_col = 'Exam_Score'
num_cols = ['Hours_Studied','Attendance','Sleep_Hours','Previous_Scores','Tutoring_Sessions','Physical_Activity']
ord_cols = ['Parental_Involvement','Access_to_Resources','Motivation_Level',
            'Family_Income','Teacher_Quality','Peer_Influence','Parental_Education_Level','Distance_from_Home']
bin_cols = ['Extracurricular_Activities', 'Internet_Access', 'School_Type',
            'Learning_Disabilities', 'Gender']

In [10]:

ordinal_categories = [
    ['Low', 'Medium', 'High'],                    # Parental_Involvement
    ['Low', 'Medium', 'High'],                    # Access_to_Resources
    ['Low', 'Medium', 'High'],                    # Motivation_Level
    ['Low', 'Medium', 'High'],                    # Family_Income
    ['Low', 'Medium', 'High'],                    # Peer_Influence
    ['Negative', 'Neutral', 'Positive'],          # Teacher_Quality
    ['High School', 'College', 'Postgraduate'],   # Parental_Education_Level
    ['Near', 'Moderate', 'Far']                   # Distance_from_Home
               
]

binary_categories = [
    ['No', 'Yes'],      # Extracurricular_Activities
    ['No', 'Yes'],      # Internet_Access
    ['Public', 'Private'],  # School_Type
    ['No', 'Yes'],      # Learning_Disabilities
    ['Female', 'Male']  # Gender
]

numerical_transformer = Pipeline(steps=[('scaler', StandardScaler())])
ordinal_transformer = Pipeline(steps=[('encoder', OrdinalEncoder(categories=ordinal_categories))])
binary_transformer = Pipeline(steps=[ ('encoder', OrdinalEncoder(categories=binary_categories))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('ord', ordinal_transformer, ord_cols),
        ('bin', binary_transformer, bin_cols)
    ],
    remainder='drop'
)



In [11]:
pipeline_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])


pipeline_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(random_state=42))
])

pipeline_lasso = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(random_state=42, max_iter=10000))
])

pipeline_enet = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ElasticNet(random_state=42, max_iter=10000))
])

Se aprovecha el uso de la función Pipeline de sklearn para la simplificación de la realización de modelos.

### Modelos

Versiones a comparar de cada modelo

| Versión | Método | Penalización |
|---|---|---|
| Base | `LinearRegression` | Ninguna |
| Ridge | `Ridge` | $L_2$ |
| Lasso | `Lasso` | $L_1$ |
| ElasticNet | `ElasticNet` | $L_1 + L_2$ |

Para Ridge, Lasso y ElasticNet, el hiperparámetro de penalización se seleccionará mediante validación cruzada de 5 folds sobre el conjunto de entrenamiento, usando `GridSearchCV` y `R²` como métrica.

#### **Regresión no lineal múltiple sin interacciones**

**Justificación**

El segundo modelo propuesto es una **regresión lineal no múltiple sin interacciones ni términos polinomiales**.

- Se buscara mejorar el *baseline* para la predicción de `Exam_Score`
- Nos permite evaluar cuales de los factores númericos tienen una relación no lineal con la calificación.
- Sus residuos permitirán diagnosticar si el modelo mejora sobre la linealidad
- Mantiene todos los efectos del modelo base y añade `Previous_Scores²`.

Para el estudiante $i$, el modelo se plantea como:

$$
\text{Exam Score}_i = \beta_0 + \sum_{j=1}^{p} \beta_j X_{ij} + \varepsilon_i
$$

donde $X_{ij}$ representa cada uno de los factores incluidos.

In [12]:
pipeline_b_m2 = pipeline_base
pipeline_r_m2 = pipeline_ridge
pipeline_l_m2 = pipeline_lasso
pipeline_en_m2 = pipeline_enet

df_m2 = df.copy()
df_m2['Previous_Scores_2'] = df_m2['Previous_Scores']**2

X_m2 = df_m2.drop(columns=[target_col])
Y_m2 = df_m2[target_col]
X_train_m2, X_test_m2, Y_train_m2, Y_test_m2 = train_test_split(X_m2, Y_m2, test_size=0.2, random_state=42)


- James, G., Witten, D., Hastie, T., Tibshirani, R. y Taylor, J. (2023). *An Introduction to Statistical Learning: with Applications in Python*. Springer. Capítulos 3, 6 y 7.
- Laboratorio de regresión 1. Regresión lineal simple, residuos, RSS, RSE y $R^2$.
- Laboratorio de regresión 2. Significancia de factores, errores estándar, pruebas de hipótesis, *p-values*, regresión múltiple e interacciones.
- Laboratorio de regresión 3. Exploración, limpieza y validación de factores del conjunto de datos.
- Laboratorio de regresión 4. Modelos penalizados, Ridge, Lasso, selección de $\alpha$ y comparación de desempeño.
- Zou, H. y Hastie, T. (2005). “Regularization and Variable Selection via the Elastic Net”. *Journal of the Royal Statistical Society: Series B*, 67(2), 301-320.